# EDA

*Análisis univariado, bivariado, en el tiempo, de atípicos y geográfico: qué factores explican el nivel de ventas*

In [ ]:
# Librerías y funciones compartidas del libro (common.py y model/features.py)
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from common import (CATEGORY_COLORS, COLORWAY, MARK, PASTEL, PROY, REAL, SEQ_SCALE, load_data,
                    load_mensual, md, show, style_fig, tabla)
from model.features import agregar_variables

# Nombres cortos de los meses y dataset por línea de pedido
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
df = load_data()


def mean_markers(d, col, y):
    """Punto sobre la media de cada grupo (equivalente a stat_summary de ggplot)."""
    return [go.Scatter(x=[c], y=[v], mode="markers", showlegend=False,
                       marker=dict(color=MARK, size=9, line=dict(width=1, color="white")))
            for c, v in d.groupby(col)[y].mean().items()]

## 1 · Análisis univariado

### Resumen estadístico de Sales

In [ ]:
# Resumen estadístico de las ventas por línea de pedido
s = df.Sales
# Cuartiles: se usan también para el rango intercuantil (RIC) y los atípicos
q1, q3 = s.quantile(0.25), s.quantile(0.75)
tabla(pd.DataFrame([{
    "n": f"{len(s):,}", "media": f"{s.mean():.0f}", "ds": f"{s.std():.0f}",
    "mediana": f"{s.median():.1f}", "q1": f"{q1:.1f}", "q3": f"{q3:.0f}",
    "RIC": f"{q3 - q1:.0f}", "minimo": f"{s.min():.1f}", "maximo": f"{s.max():.0f}",
}]))

La media (230) es 4 veces la mediana (54.5) y la desviación estándar (623) supera a la media, lo que
confirma una fuerte asimetría positiva: la mayoría de los pedidos son de bajo valor y unos pocos
pedidos muy grandes (hasta 22638) jalan el promedio hacia arriba. El RIC de 193 (entre Q1=17.3 y
Q3=210) muestra que el 50% central de los pedidos se mueve en un rango moderado.

### Histograma de Sales

In [ ]:
# Histograma hasta $1.000 (la mayoría de los pedidos); los más grandes estirarían el eje
fig = px.histogram(df[df.Sales <= 1000], x="Sales", nbins=40, color_discrete_sequence=[PASTEL["peach"]])
fig.update_traces(marker_line_width=0)
fig.update_xaxes(title="Ventas")
fig.update_yaxes(title="Número de pedidos", title_standoff=12, automargin=True)
show(style_fig(fig, 320))

Confirma visualmente el sesgo a la derecha: la gran mayoría de las barras se concentran en valores
bajos de venta, con una cola larga y dispersa hacia la derecha.

### Densidad de log(Sales)

In [ ]:
# Densidad del logaritmo de las ventas: corrige la asimetría y deja ver la forma
fig = px.histogram(df, x=np.log(df.Sales), nbins=50, histnorm="probability density",
                   color_discrete_sequence=[PASTEL["pink"]])
fig.update_traces(marker_line_width=0)
fig.update_xaxes(title="Log(Ventas)")
fig.update_yaxes(title="Densidad", title_standoff=12, automargin=True)
show(style_fig(fig, 320))

Al aplicar el logaritmo, la distribución se vuelve más simétrica y manejable. La mayor concentración
de observaciones se encuentra aproximadamente entre log(Sales) = 2.5 y 4, donde está el máximo de la
densidad. Hacia la derecha la densidad disminuye, con un pequeño abultamiento cerca de 5, mostrando
una menor concentración de ventas altas.

In [ ]:
# Barras con el número y el % de pedidos de cada valor de una variable
def barras(col, xlabel, color):
    t = df[col].value_counts().rename_axis(col).reset_index(name="n")
    t["pct"] = t.n / t.n.sum() * 100
    t = t.sort_values("n")
    fig = px.bar(t, x=col, y="n", text=t.apply(lambda r: f"{r.n:,} ({r.pct:.1f}%)", axis=1),
                 color_discrete_sequence=[PASTEL[color]])
    fig.update_traces(textposition="outside", marker_line_width=0)
    fig.update_xaxes(title=xlabel)
    fig.update_yaxes(title="Número de pedidos")
    show(style_fig(fig, 320))

### Barras: pedidos por categoría

In [ ]:
# Pedidos por categoría
barras("Category", "Categoría", "blue")

Office Supplies concentra el 60.3% de los pedidos (6026), muy por encima de Furniture (21.2%, 2121) y
Technology (18.5%, 1847). Es decir, Office Supplies es la categoría que más rota en número de pedidos.

### Barras: pedidos por región

In [ ]:
# Pedidos por región
barras("Region", "Región", "green")

West tiene mayor numero de pedidos con 3203 pedidos, seguida de East con 2848. Central con 2323 y
South con menor numero de pedidos con 1620.

## 2 · Análisis bivariado

In [ ]:
# Boxplot de las ventas por grupo, con un punto en la media; log=True usa log(ventas)
def box(col, xlabel, log=False):
    d = df.assign(log_sales=np.log(df.Sales))
    y = "log_sales" if log else "Sales"
    # Categorías con su color fijo; otras variables con la paleta general
    seq = None if col == "Category" else COLORWAY
    cmap = CATEGORY_COLORS if col == "Category" else None
    fig = px.box(d, x=col, y=y, color=col, color_discrete_map=cmap, color_discrete_sequence=seq)
    fig.add_traces(mean_markers(d, col, y))
    fig.update_layout(showlegend=False)
    fig.update_xaxes(title=xlabel)
    fig.update_yaxes(title="Log(Ventas)" if log else "Ventas")
    show(style_fig(fig, 340))

### Boxplot: log(Sales) por categoría

In [ ]:
# En log porque Technology tiene pedidos mucho más grandes que el resto
box("Category", "Categoría", log=True)

Furniture (5.205) con la mediana mas alta junto con Technology (5.113). Office Supplies queda muy por
debajo (3.311), casi 2 puntos log menos que las otras dos.

En el ancho de la caja (RIC), Technology (1.886) y Office Supplies (1.916) son muy parecidas, y
Furniture es la que tiene la caja más ancha/dispersa (2.225).

### Boxplot: Sales por región

In [ ]:
# Ventas por región
box("Region", "Región")

Las cajas de las cuatro regiones son casi iguales; lo que cambia son los valores atípicos, con
pedidos puntuales muy grandes en South y Central. A diferencia de Category, las diferencias entre
regiones son menos marcadas.

### Boxplot: Sales por segmento

In [ ]:
# Ventas por segmento de cliente
box("Segment", "Segmento")

Medianas: Consumer 53.72, Corporate 56.54, Home Office 52.44, ningun segmento sobresale.

Medias: Consumer 223.73, Corporate 233.82, Home Office 240.9, aquí Home Office tiene la media más
alta.

Consumer tiene 5191 pedidos, casi el triple que Home Office (1783). Consumer genera más ventas
totales por tener más pedidos.

## 3 · Ventas en el tiempo

### Evolución mensual de ventas

In [ ]:
# Ventas totales de cada mes (fecha del pedido llevada al primer día del mes)
t = (df.assign(mes=df.Order_Date.dt.to_period("M").dt.to_timestamp())
       .groupby("mes", as_index=False).Sales.sum())
fig = px.scatter(t, x="mes", y="Sales", color_discrete_sequence=[PASTEL["pink"]])
fig.update_traces(marker=dict(size=8, line=dict(width=1, color="white")))
fig.update_xaxes(title=None)
fig.update_yaxes(title="Ventas")
show(style_fig(fig, 340))

Los puntos muestran un patrón que se repite cada año: las ventas son más bajas a comienzos de año y
suben hacia el final, lo que sugiere un comportamiento estacional.

### Tendencia: promedio móvil de 12 meses

In [ ]:
# Promedio móvil de 12 meses: quita la estacionalidad y deja ver la tendencia
t = t.assign(movil=t.Sales.rolling(12).mean())
fig = go.Figure([
    go.Scatter(x=t.mes, y=t.Sales, mode="markers", name="Ventas del mes",
               marker=dict(size=8, color=PASTEL["blue"], line=dict(width=1, color="white"))),
    go.Scatter(x=t.mes, y=t.movil, mode="lines", name="Promedio móvil de 12 meses",
               line=dict(width=3, color=PROY)),
])
fig.update_yaxes(title="Ventas", tickprefix="$", tickformat=",.0f")
show(style_fig(fig, 340))

El promedio móvil de 12 meses quita la estacionalidad y deja ver la tendencia: se mantiene casi
plano, en torno a \$40.000 por mes, entre 2014 y 2015 y sube con claridad desde 2016, hasta unos
\$61.000 a fines de 2017. Las ventas crecieron en la segunda mitad del periodo, más que con una
tendencia lineal constante.

### Estacionalidad: % de las ventas del año por mes

In [ ]:
# Ventas por año (filas) y mes (columnas)
p = df.assign(anio=df.Order_Date.dt.year, mes=df.Order_Date.dt.month).pivot_table(
    index="anio", columns="mes", values="Sales", aggfunc="sum")
# Cada mes como % de las ventas de su año: compara la forma del año sin importar el crecimiento
pct = p.div(p.sum(axis=1), axis=0) * 100
fig = go.Figure(go.Heatmap(z=pct.values, x=MESES, y=[str(a) for a in pct.index], colorscale=SEQ_SCALE,
                           text=pct.round(1).values, texttemplate="%{text}", xgap=3, ygap=3,
                           hovertemplate="%{y} · %{x}: %{z:.1f} % del año<extra></extra>",
                           colorbar=dict(title="% del año")))
fig.update_yaxes(autorange="reversed")
show(style_fig(fig, 340))

Cada fila suma el 100 % de las ventas del año. El patrón se repite los cuatro años: enero y febrero
son los meses más bajos (entre 1 % y 6 % del año cada uno) y septiembre, noviembre y diciembre los
más altos (entre 11 % y 17 %). Por eso el modelo usa el mes del año y las ventas del mismo mes del
año anterior.

### Tendencia de ventas por categoría

In [ ]:
# Ventas mensuales por categoría (una columna por categoría)
tc = (df.assign(mes=df.Order_Date.dt.to_period("M").dt.to_timestamp())
        .groupby(["Category", "mes"]).Sales.sum().unstack(0))
fig = go.Figure()
# Puntos: cada mes · línea: promedio móvil de 12 meses
for cat in tc.columns:
    color = CATEGORY_COLORS[cat]
    fig.add_trace(go.Scatter(x=tc.index, y=tc[cat], mode="markers", marker=dict(size=6, color=color, opacity=.5),
                             showlegend=False, hovertemplate="%{y:$,.0f}<extra>" + cat + "</extra>"))
    fig.add_trace(go.Scatter(x=tc.index, y=tc[cat].rolling(12).mean(), mode="lines", name=cat,
                             line=dict(width=3, color=color),
                             hovertemplate="%{y:$,.0f}<extra>" + cat + " (promedio móvil)</extra>"))
fig.update_yaxes(title="Ventas", tickprefix="$", tickformat=",.0f")
show(style_fig(fig, 340))

Las líneas son el promedio móvil de 12 meses de cada categoría. Las tres crecen hacia el final del
periodo, y Technology tiene el nivel más alto en 27 de los 37 meses, coherente con lo visto antes:
menos pedidos pero de mayor valor. Office Supplies es la que más acelera en 2017.

## 4 · Atípicos y relación con el modelo

### Atípicos: su peso en pedidos y en ventas

In [ ]:
lim = q3 + 1.5 * (q3 - q1)  # regla del rango intercuantil
# Atípicos: pedidos por encima del límite; su % en número de pedidos y en ventas
out = s > lim
pct_ped, pct_ven = out.mean() * 100, s[out].sum() / s.sum() * 100
grupos = ["Pedidos", "Ventas (USD)"]
# Barras apiladas: normales vs. atípicos, en pedidos y en ventas
fig = go.Figure([
    go.Bar(x=grupos, y=[100 - pct_ped, 100 - pct_ven], name=f"Normales (≤ ${lim:,.0f})", marker_color=REAL,
           text=[f"{100 - pct_ped:.1f} %", f"{100 - pct_ven:.1f} %"], textposition="inside"),
    go.Bar(x=grupos, y=[pct_ped, pct_ven], name=f"Atípicos (> ${lim:,.0f})", marker_color=PROY,
           text=[f"{pct_ped:.1f} %", f"{pct_ven:.1f} %"], textposition="inside"),
])
fig.update_layout(barmode="stack")
fig.update_yaxes(title="% del total", ticksuffix=" %", range=[0, 100])
show(style_fig(fig, 340))

In [ ]:
# Lectura del gráfico escrita con las cifras calculadas
md(f"Con la regla del rango intercuantil (atípico si supera Q3 + 1.5 × RIC = ${lim:,.0f}), "
   f"{out.sum():,} pedidos son atípicos: el {pct_ped:.1f} % de los pedidos, pero el {pct_ven:.0f} % de "
   f"las ventas. No son errores de captura sino los pedidos grandes que sostienen el negocio, por eso "
   f"el ETL no los elimina. Para el modelo, estos pedidos explican los picos de algunos meses que "
   f"ninguna variable puede anticipar, y por eso se trabaja con ventas mensuales y no por pedido.")

### Dispersión: el mes contra el mismo mes del año anterior

In [ ]:
# Índice del mes = ventas del mes ÷ promedio de los 12 meses anteriores (la variable que proyecta el modelo)
mm = agregar_variables(load_mensual()).dropna(subset=["indice", "r_lag_12", "r_lag_1"])
# Correlación del índice con el del mismo mes del año anterior y con el del mes anterior
r12, r1 = mm.indice.corr(mm.r_lag_12), mm.indice.corr(mm.r_lag_1)
# Recta de mínimos cuadrados para la línea de tendencia
pend, inter = np.polyfit(mm.r_lag_12, mm.indice, 1)
xs = np.array([mm.r_lag_12.min(), mm.r_lag_12.max()])
# Un color por categoría
fig = go.Figure([go.Scatter(x=g.r_lag_12, y=g.indice, mode="markers", name=cat,
                            marker=dict(size=7, color=CATEGORY_COLORS[cat], opacity=.75,
                                        line=dict(width=1, color="white")),
                            hovertemplate="%{x:.2f} → %{y:.2f}<extra>" + cat + "</extra>")
                 for cat, g in mm.groupby("Category")])
fig.add_trace(go.Scatter(x=xs, y=inter + pend * xs, mode="lines", name="Tendencia (mínimos cuadrados)",
                         line=dict(color=PROY, width=3)))
fig.update_xaxes(title="Índice del mismo mes del año anterior")
fig.update_yaxes(title="Índice del mes")
show(style_fig(fig, 340))

In [ ]:
# Lectura del gráfico escrita con las cifras calculadas
md(f"Cada punto es una categoría en una región durante un mes (2015–2017), en índice: ventas del mes "
   f"÷ promedio de los 12 meses anteriores. La línea sube: un mes que vendió por encima de lo normal "
   f"el año pasado tiende a repetirlo (correlación {r12:.2f}). Con el mes anterior la correlación es "
   f"casi nula ({r1:.2f}). Es la relación que usa el modelo, y anticipa lo que muestra su importancia "
   f"de variables: pesa el mismo mes del año anterior, no el mes anterior. La nube es ancha, por eso "
   f"el error por serie es alto.")

## 5 · Geografía y otros factores de las ventas

In [ ]:
# Estados de EE. UU.: el mapa de Plotly los ubica por su abreviatura de dos letras
ABREV = {
    "Alabama": "AL", "Arizona": "AZ", "Arkansas": "AR", "California": "CA", "Colorado": "CO", "Connecticut": "CT",
    "Delaware": "DE", "District of Columbia": "DC", "Florida": "FL", "Georgia": "GA", "Idaho": "ID", "Illinois": "IL",
    "Indiana": "IN", "Iowa": "IA", "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME",
    "Maryland": "MD", "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV", "New Hampshire": "NH", "New Jersey": "NJ",
    "New Mexico": "NM", "New York": "NY", "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH",
    "Oklahoma": "OK", "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT", "Vermont": "VT", "Virginia": "VA",
    "Washington": "WA", "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY",
}
# Escala divergente para la correlación: rojo negativa, blanco 0, azul positiva
DIV_SCALE = [[0, "#D1574A"], [0.5, "#F4F4F4"], [1, "#2E6F95"]]

# Por estado: ventas totales, venta mediana por línea y número de líneas
st = df.groupby("State").agg(Sales=("Sales", "sum"), mediana=("Sales", "median"), lineas=("Sales", "size"))
# % del total de ventas y abreviatura del estado para el mapa
st["share"] = st.Sales / st.Sales.sum() * 100
st["code"] = st.index.map(ABREV)
st = st.sort_values("Sales", ascending=False)


# Mapa de EE. UU. coloreado por la columna col; el recuadro al pasar el cursor muestra las 3 cifras
def mapa(t, col, formato):
    fig = go.Figure(go.Choropleth(
        locations=t.code, z=t[col], locationmode="USA-states", colorscale=SEQ_SCALE,
        marker_line_color="white", marker_line_width=.6, text=t.index,
        customdata=np.stack([t.Sales, t.mediana, t.lineas], axis=1),
        hovertemplate="<b>%{text}</b><br>Ventas totales: $%{customdata[0]:,.0f}<br>Venta mediana por línea: "
                      "$%{customdata[1]:,.1f}<br>Líneas de pedido: %{customdata[2]:,}<extra></extra>",
        colorbar=dict(title="USD", tickformat=formato, thickness=12, len=.8),
    ))
    fig.update_geos(scope="usa", showlakes=False, bgcolor="rgba(0,0,0,0)", landcolor="#C9CED8")  # gris: estados sin dato
    fig = style_fig(fig, 380)
    fig.update_layout(margin=dict(l=0, r=0, t=10, b=0))
    return fig

### Mapa: ventas totales por estado

In [ ]:
# Los 4 estados que más venden
top = st.head(4)
# Mapa de ventas totales por estado
show(mapa(st, "Sales", "$,.0s"))
md(f"Ventas totales (Sales) por estado. Se concentran en pocos estados: {', '.join(top.index)} suman "
   f"el {top.share.sum():.0f} % del total, y solo California aporta el {top.share.iloc[0]:.0f} %. "
   f"El estado explica mucho del total vendido, sobre todo por el número de pedidos.")

### Mapa: venta mediana por línea de pedido

In [ ]:
# La mediana con pocas líneas no es estable: solo se pintan los estados con 30 líneas o más
ok = st[st.lineas >= 30]
grandes = st.head(3)
show(mapa(ok, "mediana", "$,.0f"))
md(f"Venta mediana por línea de pedido en cada estado (solo los {len(ok)} con 30 líneas o más). Va de "
   f"${ok.mediana.min():.0f} a ${ok.mediana.max():.0f}: el valor de cada compra cambia poco entre "
   f"estados. Los que más venden en total no son los de compras más grandes "
   f"({', '.join(f'{e} ${m:.0f}' for e, m in grandes.mediana.items())}): su ventaja es el volumen. "
   f"Por eso el modelo trabaja por región y no por estado.")

### Mapa de calor: correlación de Sales con cantidad y descuento

In [ ]:
# Variables numéricas de cada línea de pedido, con nombre en español
cols = {"Sales": "Ventas", "Quantity": "Cantidad", "Discount": "Descuento"}
# Matriz de correlación de Pearson
c = df[list(cols)].corr().rename(index=cols, columns=cols)
fig = go.Figure(go.Heatmap(z=c.values, x=c.columns, y=c.index, colorscale=DIV_SCALE, zmin=-1, zmax=1,
                           text=c.round(2).values, texttemplate="%{text}", textfont=dict(size=14),
                           hovertemplate="%{y} vs. %{x}: %{z:.2f}<extra></extra>", colorbar=dict(thickness=12)))
fig.update_yaxes(autorange="reversed")
show(style_fig(fig, 340))
# Venta mediana según el número de unidades, para el texto
med = df.groupby("Quantity").Sales.median()
md(f"Correlación de Pearson de Sales con las otras variables numéricas de cada línea de pedido. Con la "
   f"cantidad es positiva pero débil ({c.loc['Ventas', 'Cantidad']:.2f}): la venta mediana sube de "
   f"${med.loc[1]:.0f} con 1 unidad a ${med.loc[5]:.0f} con 5, pero el precio del producto pesa más "
   f"que las unidades. Con el descuento es prácticamente nula ({c.loc['Ventas', 'Descuento']:.2f}). "
   f"Profit no se incluye porque es una consecuencia de la venta, no un factor que la explique.")

### Contenedores 2D: descuento vs. ventas por línea

In [ ]:
# Contenedores 2D: ventas en escala log y color en log10 del conteo, para que unas pocas celdas no tapen al resto
xb = np.arange(0, 0.851, 0.05)          # descuento en tramos de 5 puntos
yb = np.logspace(-0.5, 4.5, 26)         # ventas de ~$0.3 a ~$32,000 en tramos logarítmicos
# Cuenta las líneas de pedido en cada celda descuento × ventas
conteo, xe, ye = np.histogram2d(df.Discount, df.Sales, bins=[xb, yb])
fig = go.Figure(go.Heatmap(
    x=(xe[:-1] + xe[1:]) / 2 * 100, y=np.sqrt(ye[:-1] * ye[1:]), z=np.log10(np.where(conteo > 0, conteo, np.nan)).T,
    customdata=conteo.T, colorscale=SEQ_SCALE, zmin=0,
    colorbar=dict(title="Líneas", thickness=12, tickvals=[0, 1, 2, 3], ticktext=["1", "10", "100", "1.000"]),
    hovertemplate="Descuento ≈ %{x:.0f} %<br>Ventas ≈ $%{y:,.0f}<br>%{customdata:,.0f} líneas<extra></extra>"))
fig.update_xaxes(title="Descuento aplicado (%)", ticksuffix=" %")
fig.update_yaxes(title="Ventas por línea (USD, escala log)", type="log", tickprefix="$",
                 tickvals=[1, 10, 100, 1000, 10000], ticktext=["1", "10", "100", "1,000", "10,000"])
show(style_fig(fig, 360))
# Líneas con descuento de 0 a 20 %, para comparar su venta mediana con las de más descuento
bajo = df.Discount <= 0.2
md(f"Cada celda cuenta las líneas de pedido con ese descuento y ese valor de venta; las ventas van en "
   f"escala logarítmica para ver todo el rango. El {bajo.mean() * 100:.0f} % de las líneas tiene "
   f"descuentos de 0 a 20 %, donde están las celdas más oscuras, y en cada columna de descuento las "
   f"ventas cubren casi el mismo rango. La venta mediana es ${df[bajo].Sales.median():.0f} con descuento "
   f"bajo y ${df[~bajo].Sales.median():.0f} con descuento alto: el descuento no hace que se venda más "
   f"por línea.")

### Burbujas: volumen y valor de las ventas por subcategoría

In [ ]:
# Por subcategoría: ventas totales, número de líneas y venta mediana
sc = df.groupby(["Category", "Sub-Category"]).agg(Sales=("Sales", "sum"), lineas=("Sales", "size"),
                                                  mediana=("Sales", "median")).reset_index()
sc["share"] = sc.Sales / sc.Sales.sum() * 100
# Una burbuja por subcategoría: x = volumen, y = valor, tamaño = ventas totales
fig = go.Figure([go.Scatter(x=g.lineas, y=g.mediana, mode="markers+text", name=cat, text=g["Sub-Category"],
                            textposition="top center", textfont=dict(size=11),
                            marker=dict(size=np.sqrt(g.Sales) / 9, color=CATEGORY_COLORS[cat], opacity=.8,
                                        line=dict(width=1, color="white")),
                            customdata=np.stack([g.Sales, g.share], axis=1),
                            hovertemplate="<b>%{text}</b><br>Líneas %{x:,}<br>Venta mediana $%{y:,.0f}"
                                          "<br>Ventas totales $%{customdata[0]:,.0f} (%{customdata[1]:.1f} %)"
                                          "<extra>" + cat + "</extra>")
                 for cat, g in sc.groupby("Category")])
fig.update_xaxes(title="Líneas de pedido (volumen)")
fig.update_yaxes(title="Venta mediana por línea (USD, escala log)", type="log", tickprefix="$",
                 tickvals=[10, 30, 100, 300, 1000], ticktext=["10", "30", "100", "300", "1,000"])
show(style_fig(fig, 440))
# Subcategorías ordenadas por ventas, para el texto
top = sc.sort_values("Sales", ascending=False)
b = sc.set_index("Sub-Category")
md(f"Gráfico de burbujas: cada una es una subcategoría y su tamaño son sus ventas totales. "
   f"{top['Sub-Category'].iloc[0]} y {top['Sub-Category'].iloc[1]} lideran (cada una con el "
   f"{top.share.iloc[0]:.0f} % del total) combinando volumen y valor. Binders tiene el mayor volumen "
   f"({b.lineas['Binders']:,} líneas) pero una mediana de solo ${b.mediana['Binders']:.0f}, y Copiers lo "
   f"contrario: {b.lineas['Copiers']} líneas con una mediana de ${b.mediana['Copiers']:,.0f}. Qué "
   f"producto se vende es el factor que más diferencia el valor de cada línea.")